# Fill-Mask

> Everything to know about masked language modelling: the objective that built the encoder era, the tokenizer traps that make `[MASK]` predictions confusing, how to use it to probe what a model learned, and runnable code comparing BERT, RoBERTa and ModernBERT on the same masked text.

- skip_showdoc: true
- skip_exec: true

## 1. What is Fill-Mask?

Fill-mask - masked language modelling (MLM) - hides tokens in a sentence and asks the model to recover them from **both sides** of context:

```
The capital of France is [MASK].  ->  Paris (0.87), Lyon (0.03), Marseille (0.02), ...
```

It is unusual among the tasks in this folder in that it is mostly **not a product**. It is the *pretraining objective* that produced BERT and every encoder descended from it, and the reason it appears as a task at all is that the pretraining head is still attached to the released checkpoints, so you can query it directly.

**Why that is useful anyway:**

- **Diagnostics.** What a model predicts into a mask is the most direct window into what pretraining taught it - facts, associations, and biases alike (sections 11 and 12).
- **Domain adaptation.** Continued MLM pretraining on your own corpus, before fine-tuning, is the cheapest reliable way to specialise an encoder for legal, biomedical or internal-jargon text.
- **Scoring.** Pseudo-perplexity from an MLM gives a usable "how natural is this sentence" score - used in grammatical-error detection and reranking.
- **Augmentation.** Mask and resample words to generate paraphrases for training data.

**MLM vs causal LM**, the distinction that decides which model you want:

| | Masked LM (BERT-style) | Causal LM (GPT-style) |
|---|---|---|
| Sees | Left **and** right context | Left context only |
| Objective | Recover masked tokens (~15% of them) | Predict the next token (all of them) |
| Training signal | ~15% of tokens per pass | 100% of tokens per pass |
| Natural at | Understanding: classification, NER, retrieval, extraction | Generating text |
| Cannot | Generate fluently | See the future |

Bidirectionality is precisely why encoders remain the better feature extractors for classification and retrieval per parameter - `01_Token_Classification`, `03_Question_Answering`, `07_Feature_Extraction` and `11_Text_Ranking` in this folder all run on models trained by the objective in this notebook.

---

## 2. Real-World Use Cases

| Use case | Domain | Consumes / produces | Dominant constraint |
|---|---|---|---|
| Domain adaptation before fine-tuning | Legal, biomedical, finance | Unlabelled in-domain corpus -> better encoder | Corpus size; GPU hours; vocabulary coverage |
| Spelling and grammar correction | Writing tools, search | Sentence -> flagged token + suggestions | Precision (false flags annoy); latency |
| Query expansion and correction | Search | Misspelled query -> corrected terms | Sub-10 ms; must not "correct" real product names |
| Data augmentation | Any ML team with little data | Sentence -> paraphrases by resampling masks | Label preservation; diversity |
| Model auditing and bias measurement | Responsible AI | Templated prompts -> prediction distributions | Choosing templates that measure what you claim |
| Knowledge probing | Research | Cloze statements -> factual recall | Prompt sensitivity; probes measure phrasing as much as knowledge |
| Sentence acceptability scoring | Linguistics, ASR reranking | Sentence -> pseudo-perplexity | Cost: one forward pass per token |
| Vocabulary extension | Domain NLP | New tokens -> embeddings initialised by MLM | Tokenizer surgery; re-training the head |

What the leaderboard number hides:

- **A high mask-recovery accuracy is not a good model.** It heavily reflects how the tokenizer splits words: a vocabulary that fragments rare words into common subwords makes each piece trivially predictable. Never compare raw fill-mask accuracy across tokenizers.
- **The prediction distribution is a description of the pretraining corpus, not of the world.** That is what makes it a good audit instrument and a bad knowledge base.
- **Probe results move with phrasing.** "The capital of France is [MASK]" and "[MASK] is the capital city of France" give different answers from the same model. Any claim from a single template is fragile; use many and report the spread.
- **MLM heads are often stripped.** Many fine-tuned checkpoints (a classifier, a retriever) discard the MLM head, and loading one for fill-mask gives an untrained head and nonsense output. Check that `AutoModelForMaskedLM` loads without warning you about newly initialised weights.

---

## 3. How Modern Masked Language Modelling Works

1. **word2vec's CBOW (2013).** Predict a word from its context. The same idea, without deep contextualisation - a single vector per word type.
2. **BERT (2018).** Mask 15% of tokens (80% replaced with `[MASK]`, 10% with a random token, 10% left alone - so the model cannot assume a masked position is the only one worth attending to), plus next-sentence prediction. Bidirectional transformers, and the start of "pretrain then fine-tune" as the default recipe.
3. **RoBERTa (2019).** Same architecture, better training: drop NSP, use dynamic masking (a fresh mask pattern every epoch), much more data, longer training, bigger batches. It showed that BERT was substantially undertrained, and its recipe is why so many "BERT" deployments are actually RoBERTa.
4. **ELECTRA (2020).** Replace masking with **replaced-token detection**: a small generator swaps some tokens, and the main model classifies *every* position as original or replaced. Learning from 100% of positions instead of 15% makes it far more sample-efficient - the objective DeBERTa-v3 later adopted.
5. **DeBERTa / DeBERTa-v3 (2020-2021).** Disentangled attention (content and relative position as separate vectors) plus ELECTRA-style pretraining. The strongest classic encoder, and the default backbone for extractive QA and NLI for years.
6. **Multilingual and domain variants (2019-present).** XLM-R (100 languages), and domain models trained by continuing MLM on specialised corpora: BioBERT/PubMedBERT, SciBERT, LegalBERT, FinBERT. Continued pretraining on in-domain text is still one of the highest-return moves in applied NLP.
7. **ModernBERT (Dec 2024) and the encoder revival.** Rotary embeddings, alternating local/global attention, GeGLU, unpadded batching, flash attention, 8192-token context, 2T training tokens including code, and a **higher mask rate** (30%) than BERT's 15% - the original rate turned out to be conservative. Roughly DeBERTa-v3 quality at several times the speed. EuroBERT, NeoBERT and mmBERT extended the recipe; the encoder line did not end, it was simply unfashionable for a few years.

**Where it stands (mid-2026).** For representation tasks - classification, retrieval, extraction, reranking - a modern encoder is still the right tool and ModernBERT-base is the default starting point. Fill-mask itself remains a diagnostic and a domain-adaptation objective rather than a deployed capability.

---

## 4. Evaluation Metrics

**Top-k accuracy on masked tokens.** Mask a token, ask for the model's top-k predictions, check whether the original is among them. Simple, and comparable **only within a tokenizer** - different vocabularies make different tasks out of the same sentence.

**Pseudo-perplexity (PPPL).** The MLM analogue of perplexity: mask each token in turn, score the true token, and exponentiate the mean negative log-likelihood.

$$\text{PPPL}(s) = \exp\left(-\frac{1}{|s|}\sum_{i=1}^{|s|} \log P(x_i \mid x_{\setminus i})\right)$$

It is **not** comparable to a causal model's perplexity - each prediction here sees both sides, so the numbers live on different scales entirely. It costs one forward pass per token, so it is for sentences, not corpora.

**What these do not measure.** Neither says anything about downstream quality. The only honest evaluation of an encoder is fine-tuning it on the task you care about and measuring that (GLUE-style). A model can win at mask recovery through tokenizer luck and lose at every task you want.

**For probing work**, the metrics are different again: precision@1 over a knowledge probe (LAMA), or an association-gap statistic between templated groups when measuring bias. Both are extremely prompt-sensitive - report the template set, not one sentence.

---

In [ ]:
import torch


@torch.inference_mode()
def pseudo_perplexity(model, tok, sentence):
    "Mask each token in turn, score the true token, exponentiate the mean NLL.\n\n    One forward pass per token, batched here into a single call - fine for sentences,\n    hopeless for corpora. Not comparable with a causal model's perplexity.\n    "
    enc = tok(sentence, return_tensors="pt")
    ids = enc["input_ids"][0]
    special = set(tok.all_special_ids)
    positions = [i for i, t in enumerate(ids.tolist()) if t not in special]
    if not positions:
        return float("nan")

    batch = ids.repeat(len(positions), 1)
    for row, pos in enumerate(positions):
        batch[row, pos] = tok.mask_token_id
    batch = batch.to(model.device)
    logits = model(input_ids=batch, attention_mask=torch.ones_like(batch)).logits

    nll = 0.0
    for row, pos in enumerate(positions):
        log_probs = torch.log_softmax(logits[row, pos].float(), dim=-1)
        nll -= log_probs[ids[pos]].item()
    return float(torch.exp(torch.tensor(nll / len(positions))))


@torch.inference_mode()
def topk_masked(model, tok, sentence_ids, position, k=5):
    "Top-k token predictions for one masked position. Returns [(token, probability), ...]."
    ids = sentence_ids.clone()
    ids[position] = tok.mask_token_id
    out = model(input_ids=ids.unsqueeze(0).to(model.device))
    probs = torch.softmax(out.logits[0, position].float(), dim=-1)
    top = probs.topk(k)
    return [(tok.decode([i]).strip(), p) for p, i in zip(top.values.tolist(), top.indices.tolist())]


def mask_recovery(model, tok, texts, k=5, max_length=64, seed=0):
    "Mask one random non-special token per text; report top-1 and top-k recovery accuracy.\n\n    Comparable across models ONLY if they share a tokenizer - each model is scored on its\n    own segmentation of the same strings, which is a slightly different task each time.\n    "
    generator = torch.Generator().manual_seed(seed)
    hits1 = hitsk = total = 0
    for text in texts:
        ids = tok(text, return_tensors="pt", truncation=True, max_length=max_length)["input_ids"][0]
        candidates = [i for i, t in enumerate(ids.tolist()) if t not in set(tok.all_special_ids)]
        if len(candidates) < 4:
            continue
        pos = candidates[int(torch.randint(len(candidates), (1,), generator=generator))]
        gold = tok.decode([ids[pos]]).strip()
        preds = [t for t, _ in topk_masked(model, tok, ids, pos, k=k)]
        hits1 += preds[0] == gold
        hitsk += gold in preds
        total += 1
    return {"top1": hits1 / total, f"top{k}": hitsk / total, "n": total}


print("helpers ready - pseudo_perplexity, topk_masked, mask_recovery")

## 5. Datasets

Fill-mask is evaluated on whatever text you care about; what matters is the corpus you *pretrain* on.

| Dataset | Contents | Size | Scope | License | Typical use |
|---|---|---|---|---|---|
| [WikiText-2 / 103](https://huggingface.co/datasets/Salesforce/wikitext) | Clean Wikipedia articles | 2M / 100M tokens | en | CC BY-SA 3.0 | Mask recovery and PPPL; used below |
| [C4](https://huggingface.co/datasets/allenai/c4) | Cleaned CommonCrawl | 750 GB | en (+ mC4) | ODC-By | The standard encoder pretraining corpus |
| [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) | Education-filtered web text | 1.3T tokens | en | ODC-By | Modern high-quality pretraining |
| [BookCorpus + Wikipedia](https://huggingface.co/datasets/bookcorpus/bookcorpus) | The original BERT mix | 3.3B words | en | unclear | Historical reproduction |
| [PubMed abstracts](https://huggingface.co/datasets/ncbi/pubmed) | Biomedical abstracts | 35M records | en | public domain | BioBERT/PubMedBERT-style adaptation |
| [The Pile / Dolma](https://huggingface.co/datasets/allenai/dolma) | Mixed open corpora | 825 GB / 3T tokens | en | mixed | Open pretraining |
| [LAMA probes](https://huggingface.co/datasets/facebook/lama) | Cloze statements over facts | 50k+ | en | CC BY-NC 4.0 | Knowledge probing |
| [CrowS-Pairs](https://huggingface.co/datasets/nyu-mll/crows_pairs) | Minimal sentence pairs across 9 bias types | 1.5k | en | CC BY-SA 4.0 | Bias measurement with a proper protocol |
| [StereoSet](https://huggingface.co/datasets/McGill-NLP/stereoset) | Stereotype/anti-stereotype contexts | 17k | en | CC BY-SA 4.0 | Bias measurement |

This notebook uses **WikiText-2-raw test** for mask recovery and pseudo-perplexity, read directly from its 0.7 MB parquet. For the probing sections it uses hand-written templates - fine for illustration, and explicitly *not* a measurement protocol: CrowS-Pairs and StereoSet exist because doing this properly requires controlled minimal pairs and a lot of them.

---

## 6. The Model Landscape (mid-2026)

Encoders are compared through their fine-tuned downstream scores ([GLUE](https://gluebenchmark.com/leaderboard), [MTEB](https://huggingface.co/spaces/mteb/leaderboard)), not through fill-mask accuracy.

| Model | Params | License | Context | Vocab | Notes |
|---|---|---|---|---|---|
| [bert-base-uncased](https://huggingface.co/google-bert/bert-base-uncased) | 110M | Apache 2.0 | 512 | 30k WordPiece | the original; lowercases everything; used below |
| [roberta-base](https://huggingface.co/FacebookAI/roberta-base) | 125M | MIT | 512 | 50k BPE | better-trained BERT; byte-level BPE; used below |
| [distilbert-base-uncased](https://huggingface.co/distilbert/distilbert-base-uncased) | 67M | Apache 2.0 | 512 | 30k | 2x faster, ~97% of BERT's GLUE |
| [ModernBERT-base](https://huggingface.co/answerdotai/ModernBERT-base) | 149M | Apache 2.0 | **8192** | 50k BPE | the 2026 default; used below |
| [ModernBERT-large](https://huggingface.co/answerdotai/ModernBERT-large) | 395M | Apache 2.0 | 8192 | 50k | best open encoder quality |
| [DeBERTa-v3-base](https://huggingface.co/microsoft/deberta-v3-base) | 184M | MIT | 512 | 128k | ELECTRA-style objective; **no usable MLM head** |
| [XLM-RoBERTa-base](https://huggingface.co/FacebookAI/xlm-roberta-base) | 278M | MIT | 512 | 250k | 100 languages |
| [PubMedBERT](https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext) | 110M | MIT | 512 | 30k domain | biomedical vocabulary from scratch |
| [SciBERT](https://huggingface.co/allenai/scibert_scivocab_uncased) | 110M | Apache 2.0 | 512 | 31k domain | scientific text |

**A note on DeBERTa-v3.** It is the strongest classic encoder for downstream fine-tuning, and it is deliberately not in the fill-mask comparison below: it was pretrained with replaced-token detection rather than masked-token prediction, so the released checkpoint has no meaningful MLM head. That is a good illustration of the section-2 warning - "it is an encoder" does not imply "it can fill a mask".

**How to choose.** Fine-tuning anything new in 2026: ModernBERT-base, and ModernBERT-large if the accuracy is worth 2.5x the compute. Domain-specific text with an unusual vocabulary: a domain-pretrained model, or continued MLM on your own corpus. Multilingual: XLM-R. BERT-base itself is now a teaching model rather than a recommendation.

---

## 7. Setup

Package roles:

- `transformers` (>=5.13) + `torch` - the three encoders
- `accelerate` - device placement
- `datasets` - the WikiText-2 slice
- `pandas` + `pyecharts` - benchmark table and chart

`pipeline("fill-mask")` survived the transformers v5 cleanup, so the demo path is one line. The comparison sections call `AutoModelForMaskedLM` directly, because the interesting parts are the raw probabilities and the per-model mask token.

**The tokenizer traps that make fill-mask confusing**, all of which the sections below hit deliberately:

1. **The mask token differs per model.** BERT uses `[MASK]`, RoBERTa and ModernBERT use `<mask>`. Hard-coding the string breaks silently across models - use `tok.mask_token`.
2. **The model predicts one token, not one word.** `The capital of France is [MASK].` works because `Paris` is one token. Ask for a rare multi-token word and no single prediction can be right; you need one mask per subword, and the joint prediction is not what independent masks give you.
3. **RoBERTa's byte-level BPE encodes the leading space in the token.** `" Paris"` and `"Paris"` are different tokens, and the space-prefixed one is what appears mid-sentence. Raw tokens print with a visible marker standing in for that leading space; decoding handles it, exact string comparison does not.
4. **`bert-base-uncased` lowercases its input.** Case-based clues are simply gone before the model sees anything.

---

In [ ]:
# Everything runs through Hugging Face transformers - no vendor packages.
# %pip install -q torch transformers accelerate datasets pandas pyecharts

In [ ]:
import ctypes
import ctypes.util
import gc
import time
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device != "cpu" else torch.float32
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device, "| dtype:", dtype)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:22s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "Collect garbage and hand freed VRAM back to the CUDA allocator.\n\n    Call right after `del`-ing a model you are done with: `del model; free_memory()`.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    # glibc keeps freed CPU allocations in its arenas instead of returning them to the
    # OS, so RSS compounds across sections. malloc_trim(0) hands the arenas back.
    try:
        ctypes.CDLL(ctypes.util.find_library("c") or "libc.so.6").malloc_trim(0)
    except Exception:
        pass


# All downloads go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)
HF_CACHE = str(DATA_DIR / "hf_cache")

In [ ]:
from datasets import load_dataset

wiki = load_dataset(
    "parquet",
    data_files={"test": "hf://datasets/Salesforce/wikitext/wikitext-2-raw-v1/test-*.parquet"},
    split="test",
    cache_dir=HF_CACHE,
)

# Sentences long enough to have real context, short enough to score cheaply.
SENTENCES = [t.strip() for t in wiki["text"] if 120 < len(t.strip()) < 400][:150]
print(f"{len(SENTENCES)} evaluation sentences")
for s in SENTENCES[:2]:
    print(f"\n  {s[:200]}")

## 8. The original: bert-base-uncased

110M parameters, 12 layers, a 30k WordPiece vocabulary, and the model that made "pretrain then fine-tune" the default in NLP. Its MLM head is intact, so the pretraining objective is directly queryable.

Three things the cell below demonstrates, in order:

- **A clean factual completion**, where the answer happens to be a single token.
- **The multi-token failure**, where the right answer cannot be produced by one mask no matter how good the model is - not a knowledge failure but a *representation* failure, and the reason `[MASK]` demos always use short, common words.
- **Uncased-ness**, which silently removes the capitalisation cue that would tell a cased model this is a proper noun.

---

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline

filler = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=device,
                  model_kwargs={"cache_dir": HF_CACHE})
mask = filler.tokenizer.mask_token
print("mask token:", mask, "| vocab:", filler.tokenizer.vocab_size)

for template in [
    f"The capital of France is {mask}.",
    f"A {mask} is a tool used for driving nails into wood.",
    f"The patient was prescribed {mask} for the infection.",
    f"I need to {mask} the report before the meeting.",
]:
    preds = filler(template, top_k=5)
    print(f"\n{template}")
    print("   " + "   ".join(f"{p['token_str']} {p['score']:.3f}" for p in preds))

# The multi-token limit: one mask can only ever produce one token.
tok_probe = filler.tokenizer
for word in ["Paris", "Ouagadougou", "acetaminophen"]:
    pieces = tok_probe.tokenize(word)
    print(f"\n{word!r} -> {pieces} ({len(pieces)} tokens)"
          f"{'  <- unreachable from a single mask' if len(pieces) > 1 else ''}")

del filler
free_memory()
vram("after bert pipeline")

## 9. Byte-level BPE: roberta-base

Same architecture family, better training recipe (no next-sentence prediction, dynamic masking, ten times the data), and a different tokenizer that produces a genuinely confusing artefact the first time you see it.

RoBERTa uses **byte-level BPE**, in which a leading space is part of the token. `" Paris"` and `"Paris"` are two different vocabulary entries, and the one that occurs mid-sentence is the space-prefixed one, which shows up with a placeholder glyph for the space when you print raw tokens. Practical consequences:

- Its mask token is `<mask>`, not `[MASK]`.
- Whether you write `is <mask>.` or `is<mask>.` changes which token the model is being asked for.
- Comparing a predicted string against a gold string without stripping fails silently.

Everything downstream of RoBERTa - including most sentence-embedding models - inherits this, so it is worth seeing once at the source.

---

In [ ]:
rob_tok = AutoTokenizer.from_pretrained("FacebookAI/roberta-base", cache_dir=HF_CACHE)
rob = AutoModelForMaskedLM.from_pretrained(
    "FacebookAI/roberta-base", dtype=dtype, cache_dir=HF_CACHE
).to(device).eval()
vram("roberta loaded")

print("mask token:", rob_tok.mask_token, "| vocab:", rob_tok.vocab_size)
print("tokens for 'Paris' vs ' Paris':",
      rob_tok.tokenize("Paris"), rob_tok.tokenize(" Paris"))

rob_filler = pipeline("fill-mask", model=rob, tokenizer=rob_tok, device=device)
for template in [
    f"The capital of France is {rob_tok.mask_token}.",
    f"The patient was prescribed {rob_tok.mask_token} for the infection.",
    f"Paris is the capital of {rob_tok.mask_token}.",
]:
    preds = rob_filler(template, top_k=5)
    print(f"\n{template}")
    print("   " + "   ".join(f"{p['token_str']!r} {p['score']:.3f}" for p in preds))

print("\n^ the leading spaces in those token strings are real vocabulary content, not formatting")

# Pseudo-perplexity: an MLM's answer to 'does this sentence look like English'.
for sentence in [
    "The committee approved the budget on Tuesday.",
    "The committee approve the budget on Tuesday.",
    "Committee the budget Tuesday approved on the.",
]:
    print(f"  PPPL {pseudo_perplexity(rob, rob_tok, sentence):8.2f}   {sentence}")

del rob, rob_tok, rob_filler
free_memory()
vram("after roberta")

## 10. The modern encoder: ModernBERT

ModernBERT (Dec 2024) is the BERT recipe rebuilt with everything learned since 2018: rotary position embeddings, alternating local and global attention, GeGLU activations, no padding waste, flash attention, **8192-token context**, 2 trillion training tokens including code, and a 30% masking rate rather than BERT's 15%.

What that means in practice on this hardware: several times faster than DeBERTa-v3 at similar or better downstream quality, and a context window that reads a whole document instead of its first two pages. It is the default choice for any new encoder fine-tune in 2026, and it is what `07_Feature_Extraction` uses through `gte-modernbert-base`.

Its tokenizer is a modified byte-level BPE, so the space-prefix behaviour of section 9 applies here too - but it is cased, and it knows code.

---

In [ ]:
mb_tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base", cache_dir=HF_CACHE)
mb = AutoModelForMaskedLM.from_pretrained(
    "answerdotai/ModernBERT-base", dtype=dtype, cache_dir=HF_CACHE
).to(device).eval()
vram("modernbert loaded")

print("mask token:", mb_tok.mask_token,
      "| max context:", mb.config.max_position_embeddings, "tokens")

mb_filler = pipeline("fill-mask", model=mb, tokenizer=mb_tok, device=device)
for template in [
    f"The capital of France is {mb_tok.mask_token}.",
    f"The patient was prescribed {mb_tok.mask_token} for the infection.",
    f"import numpy as {mb_tok.mask_token}",
    f"def {mb_tok.mask_token}(self, request):",
]:
    preds = mb_filler(template, top_k=5)
    print(f"\n{template}")
    print("   " + "   ".join(f"{p['token_str']!r} {p['score']:.3f}" for p in preds))

print("\n^ the last two are why 'trained on code' shows up in a fill-mask demo")

del mb, mb_tok, mb_filler
free_memory()
vram("after modernbert")

## 11. Probing what pretraining taught: facts and associations

The most useful non-production use of fill-mask: reading the pretraining corpus back out of the model.

**Factual probing** (the LAMA protocol, 2019) asks how much relational knowledge an encoder absorbed - `Dante was born in [MASK]`. It works surprisingly often, and the well-documented catch is that scores move substantially with phrasing, so a probe measures *prompt fit* and knowledge together. Never treat an encoder as a knowledge base on the strength of one template.

**Association probing** measures which completions the corpus makes likely for otherwise-identical contexts differing only in a demographic term. This is a standard fairness diagnostic - it is how BERT's occupational associations were first characterised in 2019 - and the results below are a property of the training corpus, surfaced rather than created by the model.

Two methodological warnings, because this is easy to do badly:

- **Hand-written templates are illustrations, not measurements.** A real audit uses a controlled instrument such as CrowS-Pairs or StereoSet, many minimal pairs, and a stated statistic.
- **Do not read a ranked list as an opinion.** These are corpus co-occurrence statistics conditioned on a template. The number to take away is the *gap* between matched contexts, not any single completion.

The point of running it is practical: a fine-tuned classifier inherits these associations, so if you are shipping an encoder, this is part of the due diligence.

---

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased", cache_dir=HF_CACHE)
bert = AutoModelForMaskedLM.from_pretrained(
    "google-bert/bert-base-uncased", dtype=dtype, cache_dir=HF_CACHE
).to(device).eval()
bert_filler = pipeline("fill-mask", model=bert, tokenizer=bert_tok, device=device)
m = bert_tok.mask_token

print("=== factual probing (LAMA-style) ===")
for template in [
    f"Dante was born in {m}.",
    f"The Eiffel Tower is located in {m}.",
    f"Water boils at one hundred degrees {m}.",
    f"The author of Pride and Prejudice is {m} Austen.",
]:
    preds = bert_filler(template, top_k=3)
    print(f"{template:58s} -> " + ", ".join(f"{p['token_str']} ({p['score']:.2f})" for p in preds))

print("\n=== the same fact, four phrasings - the probe's fragility ===")
for template in [
    f"The capital of Australia is {m}.",
    f"{m} is the capital of Australia.",
    f"Australia's capital city is called {m}.",
    f"The seat of the Australian government is in {m}.",
]:
    preds = bert_filler(template, top_k=3)
    print(f"{template:58s} -> " + ", ".join(f"{p['token_str']} ({p['score']:.2f})" for p in preds))

print("\n=== association probing: matched contexts, one word changed ===")
PAIRS = [
    (f"The man worked as a {m}.", f"The woman worked as a {m}."),
    (f"He is very {m}.", f"She is very {m}."),
    (f"The immigrant was described as {m}.", f"The citizen was described as {m}."),
]
for left, right in PAIRS:
    lp = [p["token_str"] for p in bert_filler(left, top_k=6)]
    rp = [p["token_str"] for p in bert_filler(right, top_k=6)]
    print(f"\n  {left}\n     {', '.join(lp)}")
    print(f"  {right}\n     {', '.join(rp)}")
    overlap = len(set(lp) & set(rp)) / len(set(lp) | set(rp))
    print(f"  top-6 overlap: {overlap:.2f}  (1.0 would mean the swapped word changed nothing)")

print("\nThese are statistics of the 2018 pretraining corpus. A classifier fine-tuned on this\n"
      "encoder inherits them, which is why this belongs in a pre-deployment checklist.")

del bert, bert_tok, bert_filler
free_memory()
vram("after probing")

## 12. Head-to-head Benchmark

Three encoders, the same 150 WikiText sentences, one masked token per sentence, one model live at a time.

**Read the accuracy column with the tokenizer column next to it.** Each model is scored on *its own* segmentation of the same strings, so a 30k-vocabulary WordPiece model and a 50k byte-level BPE model are not answering the same questions - the BPE models split rare words into more, individually easier pieces. This is the same comparability problem perplexity has in `08_Text_Generation`, and it is why nobody ranks encoders by fill-mask accuracy.

What *is* comparable and does matter: **throughput** and **pseudo-perplexity within a model** (for sentence scoring). And the real ranking - downstream fine-tuned quality - is not measurable here at all, which is worth stating plainly rather than implying the chart settles anything.

---

In [ ]:
import pandas as pd

MODELS = [
    ("bert-base-uncased", "google-bert/bert-base-uncased", 110),
    ("roberta-base", "FacebookAI/roberta-base", 125),
    ("modernbert-base", "answerdotai/ModernBERT-base", 149),
]

SCORE_SENTENCES = [
    "The committee approved the budget on Tuesday.",
    "The committee approve the budget on Tuesday.",
    "Colorless green ideas sleep furiously.",
]

results = []
for name, model_id, params_m in MODELS:
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=HF_CACHE)
    model = AutoModelForMaskedLM.from_pretrained(
        model_id, dtype=dtype, cache_dir=HF_CACHE
    ).to(device).eval()

    t0 = time.perf_counter()
    rec = mask_recovery(model, tok, SENTENCES, k=5)
    elapsed = time.perf_counter() - t0
    pppl = {s: round(pseudo_perplexity(model, tok, s), 2) for s in SCORE_SENTENCES}

    results.append({
        "model": name, "params_m": params_m, "vocab": tok.vocab_size,
        "context": model.config.max_position_embeddings,
        "top1": round(rec["top1"], 3), "top5": round(rec["top5"], 3),
        "sents_per_sec": round(rec["n"] / elapsed, 1),
        "pppl_grammatical": pppl[SCORE_SENTENCES[0]],
        "pppl_ungrammatical": pppl[SCORE_SENTENCES[1]],
    })
    print(results[-1])
    del model, tok    # free each model before loading the next so VRAM stays flat
    free_memory()

vram("after benchmark")
df = pd.DataFrame(results)
print("\naccuracy is comparable only within a tokenizer - look at the vocab column")
df

In [ ]:
from pyecharts import options as opts
from pyecharts.charts import Bar

bar = (
    Bar()
    .add_xaxis([r["model"] for r in results])
    .add_yaxis("top-1 x100", [round(r["top1"] * 100, 1) for r in results])
    .add_yaxis("top-5 x100", [round(r["top5"] * 100, 1) for r in results])
    .add_yaxis("sentences / second", [r["sents_per_sec"] for r in results])
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="Masked-token recovery on 150 WikiText sentences",
            subtitle="RTX 3060 - different tokenizers mean these are not the same task",
        ),
        yaxis_opts=opts.AxisOpts(name="value"),
        xaxis_opts=opts.AxisOpts(name="model", axislabel_opts=opts.LabelOpts(rotate=12)),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
)
bar.render_notebook()

In [ ]:
# Pseudo-perplexity as a grammaticality signal: within a model, the ungrammatical sentence
# should score higher. Across models the scale is meaningless - hence the grouped bars.
bar2 = (
    Bar()
    .add_xaxis([r["model"] for r in results])
    .add_yaxis("PPPL grammatical", [r["pppl_grammatical"] for r in results])
    .add_yaxis("PPPL ungrammatical", [r["pppl_ungrammatical"] for r in results])
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="Pseudo-perplexity separates grammatical from ungrammatical",
            subtitle="compare the two bars within a model, never across models",
        ),
        yaxis_opts=opts.AxisOpts(name="pseudo-perplexity"),
        xaxis_opts=opts.AxisOpts(name="model"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
    )
)
bar2.render_notebook()

## 13. Interactive: mask your own sentence

Write a sentence with `[MASK]` in it and see what the model expects there. This is the cell people run on its own, so it opens with a `require(...)` guard naming what it needs from Setup rather than dying on a bare `NameError`.

The experiments that teach the most:

- **Move the mask around one sentence.** Content words are hard, function words are nearly free - which is most of what mask-recovery accuracy actually measures.
- **Mask a rare or domain word** and check `tok.tokenize(word)` first. If it is more than one token, no single prediction can be right; that is the section-8 limit, not ignorance.
- **Give and remove context.** `He was born in [MASK]` versus `The Italian poet Dante was born in [MASK]`. The change in the distribution is bidirectional context doing its job.
- **Score your own sentence pairs** with pseudo-perplexity - it is a usable grammaticality signal, and a good sanity check on a domain-adapted model.

---

In [ ]:
def require(*names):
    "Fail early and clearly if the notebook's setup / helper cells have not been run."
    missing = [n for n in names if n not in globals()]
    if missing:
        raise NameError(
            f"this demo needs {', '.join(missing)} from earlier in the notebook. "
            "Run the setup and helper cells first (Run > Run All Above Selected Cell)."
        )


require("device", "dtype", "HF_CACHE", "free_memory", "vram", "pseudo_perplexity")

from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline

MODEL = "answerdotai/ModernBERT-base"   # try google-bert/bert-base-uncased for the contrast
# Write [MASK] where you want a prediction; it is rewritten to the model's own mask token.
MY_SENTENCES = [
    "The deployment failed because the database [MASK] timed out.",
    "She was born in [MASK].",
    "The Italian poet Dante was born in [MASK].",
    "Please [MASK] the pull request before Friday.",
]
MY_PAIRS = [
    ("The results were consistent with the hypothesis.",
     "The results was consistent with the hypothesis."),
]

# Re-runnable: this cell frees the model at the end, so guard the load or a second
# shift-enter raises NameError.
if "my_mlm" not in globals():
    my_tok = AutoTokenizer.from_pretrained(MODEL, cache_dir=HF_CACHE)
    my_mlm = AutoModelForMaskedLM.from_pretrained(
        MODEL, dtype=dtype, cache_dir=HF_CACHE
    ).to(device).eval()
    my_filler = pipeline("fill-mask", model=my_mlm, tokenizer=my_tok, device=device)
    vram("live model")

for sentence in MY_SENTENCES:
    text = sentence.replace("[MASK]", my_tok.mask_token)
    preds = my_filler(text, top_k=5)
    print(f"\n{sentence}")
    print("   " + "   ".join(f"{p['token_str'].strip()!r} {p['score']:.3f}" for p in preds))

print("\npseudo-perplexity (lower = more natural to this model):")
for good, bad in MY_PAIRS:
    print(f"  {pseudo_perplexity(my_mlm, my_tok, good):8.2f}  {good}")
    print(f"  {pseudo_perplexity(my_mlm, my_tok, bad):8.2f}  {bad}")

del my_mlm, my_tok, my_filler
free_memory()
vram("final")

## 14. Going Further

- **Continued pretraining is the highest-return use of this objective.** Run MLM on a few hundred MB of your own domain text before fine-tuning; it typically buys more than a larger model does, and on this hardware a ModernBERT-base adaptation is hours, not days. `DataCollatorForLanguageModeling(mlm_probability=0.3)` plus `Trainer` is the whole setup.
- **Extend the vocabulary for domain jargon.** If your key terms fragment into five subwords each, add tokens with `tokenizer.add_tokens(...)` and `model.resize_token_embeddings(...)`, then run MLM so the new embeddings learn something. Check the fragmentation first - it is often the real problem behind "the model does not understand our terminology".
- **Use MLM for augmentation carefully.** Masking and resampling generates paraphrases, and it also flips sentiment and negation without warning. Filter the results with the classifier you are augmenting for.
- **Prefer ELECTRA-style objectives if you are pretraining from scratch.** Learning from every position rather than 15% of them is a large sample-efficiency win, which is why DeBERTa-v3 adopted it.
- **Do bias measurement properly.** Section 11 is an illustration. A defensible audit uses CrowS-Pairs or StereoSet, reports a statistic over many minimal pairs, and states the limits of what a template-based measure can show.
- **Remember what encoders are for.** Fill-mask is the objective; the products are classification, extraction, retrieval and reranking. If you came here to *generate* text, `08_Text_Generation` is the notebook.
- **Related notebooks.** `00_Text_Classification` and `01_Token_Classification` (fine-tuning these encoders), `07_Feature_Extraction` and `10_Sentence_Similarity` (the same encoders as embedders), `03_Question_Answering` (span heads), `08_Text_Generation` (the causal counterpart), `04_Zero_Shot_Classification` (NLI heads on the same backbones).

---